# **Text-to-Image Generation**

Text-to-image models synthesize a novel image from a natural-language **prompt**. The state of the art is built on **diffusion models**: a network learns to *denoise* data, and generation is run as iterative denoising from pure noise, guided by the text.

**Stable Diffusion** is the canonical open model. Its key efficiency trick is **latent diffusion** — doing the expensive diffusion process in a compressed latent space rather than on full-resolution pixels.

## **1. Diffusion in one paragraph**

A **forward process** gradually adds Gaussian noise to an image over $T$ steps until it is indistinguishable from noise. A neural network (a **UNet**) is trained to predict the noise that was added at a given step $t$. At inference, we start from random noise $x_T \sim \mathcal{N}(0, I)$ and repeatedly subtract the predicted noise (a **reverse / denoising** process) to recover a clean sample. The training objective is simply

$$\mathcal{L} = \mathbb{E}_{x, t, \epsilon}\left[\; \lVert \epsilon - \epsilon_\theta(x_t, t, c) \rVert^2 \;\right]$$

where $\epsilon$ is the true noise, $\epsilon_\theta$ is the UNet's prediction, and $c$ is the text conditioning.

## **2. Stable Diffusion architecture: three components**

Latent diffusion factors the system into three trained networks:

- **Text encoder** — a frozen CLIP (or T5) text model turns the prompt into token embeddings $c$. These condition the UNet via **cross-attention**.
- **VAE (autoencoder)** — an encoder compresses a $512\times512\times3$ image into a small latent (e.g. $64\times64\times4$); a decoder reconstructs pixels from a latent. Diffusion happens *entirely in this latent space*, which is what makes it fast.
- **UNet** — the denoiser. At each step it takes the noisy latent $z_t$, the timestep $t$, and the text embedding $c$, and predicts the noise. Cross-attention layers let every spatial location attend to the prompt tokens.

Pipeline at inference: `prompt → text encoder → c`; start from noise latent; **UNet denoises over N steps** (guided by $c$); finally **VAE decoder → image**.

## **3. Classifier-free guidance (CFG)**

Naïve conditional sampling often follows the prompt weakly. **Classifier-free guidance** sharpens prompt adherence without a separate classifier. During training the text condition is randomly dropped (replaced by an empty prompt) some fraction of the time, so the same UNet learns both a *conditional* $\epsilon_\theta(z_t, c)$ and an *unconditional* $\epsilon_\theta(z_t, \varnothing)$ prediction. At inference we extrapolate away from the unconditional toward the conditional:

$$\hat{\epsilon} = \epsilon_\theta(z_t, \varnothing) + w\,\big(\epsilon_\theta(z_t, c) - \epsilon_\theta(z_t, \varnothing)\big)$$

The **guidance scale** $w$ (in diffusers, `guidance_scale`) trades off prompt fidelity vs diversity; typical values are 7–12. Higher $w$ = stronger prompt adherence but can over-saturate.

## **4. Conceptual diffusers usage**

```python
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

prompt = "a watercolor painting of a lighthouse at sunset, highly detailed"
image = pipe(
    prompt,
    num_inference_steps=30,   # number of denoising steps
    guidance_scale=7.5,       # classifier-free guidance weight
).images[0]

image.save("lighthouse.png")
```

Under the hood `pipe.text_encoder`, `pipe.unet`, and `pipe.vae` are exactly the three components above, and `pipe.scheduler` controls the denoising schedule.

**References**
- Ho et al., *Denoising Diffusion Probabilistic Models* (DDPM), 2020.
- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (Stable Diffusion), 2022.
- Ho & Salimans, *Classifier-Free Diffusion Guidance*, 2022.